In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import binomtest

In [2]:
d = np.load("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/token_scores_olmo_career.npz")
df = pd.DataFrame({k: d[k] for k in d.files})

df_selected = pd.read_csv("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/binned_intruder_candidates_career_top_bin0.0-3.0_s0.csv")
print(df.head(10))
print(df_selected.head(10))

   example_idx  position  token_id        score
0            0        74      1271  -671.944580
1            0        75      3480   592.391663
2            0        76       505  -129.717651
3            0        77     33400  -240.599060
4            0        78       311     7.384064
5            0        79      3241     0.797302
6            0        80     15009    -1.183472
7            0        81        11   289.085571
8            0        82       499  -546.612427
9            0        83       649  1281.483643
  side  intruder_bin_low  intruder_bin_high  distractor_bin_low  \
0  top               0.0                3.0                   0   
1  top               0.0                3.0                   0   
2  top               0.0                3.0                   0   
3  top               0.0                3.0                   0   
4  top               0.0                3.0                   0   
5  top               0.0                3.0                   0   
6  

In [3]:
def compute_global_percentile(scores: dict[str, np.ndarray]) -> np.ndarray:
    return (100 * (1 - pd.Series(scores["score"]).rank(pct=True))).to_numpy()

df["pctile"] = compute_global_percentile(df)
bins_edges = [0, 3, 7, 19, 50, 81, 93, 97, 100]
bin_labels = [f"{bins_edges[i]}-{bins_edges[i+1]}%" for i in range(len(bins_edges)-1)]
df["pctile_bin"] = pd.cut(df["pctile"], bins=bins_edges, labels=bin_labels, include_lowest=True)
print(df.head(10))

   example_idx  position  token_id        score     pctile pctile_bin
0            0        74      1271  -671.944580  89.412020     81-93%
1            0        75      3480   592.391663  11.312972      7-19%
2            0        76       505  -129.717651  69.085632     50-81%
3            0        77     33400  -240.599060  75.883753     50-81%
4            0        78       311     7.384064  40.481674     19-50%
5            0        79      3241     0.797302  43.135582     19-50%
6            0        80     15009    -1.183472  49.428019     19-50%
7            0        81        11   289.085571  19.307265     19-50%
8            0        82       499  -546.612427  86.802826     81-93%
9            0        83       649  1281.483643   4.155843       3-7%


In [4]:
# Sanity check 1: basic shape -- rows per distractor_bin column (should all equal
# max_per_bin, modulo any skips from an exhausted/thin pool) and the correct_label
# distribution (should be roughly uniform across A-E, same check used throughout
# this project for every candidate-selection script).
print(f"{len(df_selected)} rows")
print("\nrows per distractor_bin:")
print(df_selected.groupby(["distractor_bin_low", "distractor_bin_high"]).size())
print("\ncorrect_label distribution (should be roughly uniform across A-E):")
print(df_selected["correct_label"].value_counts().sort_index())

3200 rows

rows per distractor_bin:
distractor_bin_low  distractor_bin_high
0                   3                      400
3                   7                      400
7                   19                     400
19                  50                     400
50                  81                     400
81                  93                     400
93                  97                     400
97                  100                    400
dtype: int64

correct_label distribution (should be roughly uniform across A-E):
correct_label
A    646
B    660
C    678
D    615
E    601
Name: count, dtype: int64


In [5]:
# Sanity check 2: structural invariants -- every row's 5 candidates come from 5
# distinct documents (the whole point of the cross-document design), and no
# distractor document is reused across different intruders within the same
# distractor_bin column (the dedup fix build_rows added specifically so the
# binomial CIs per bin aren't computed over correlated draws from a shared pool).
LABELS = list("ABCDE")
example_idx_cols = [f"example_idx_{l}" for l in LABELS]

n_distinct = df_selected[example_idx_cols].nunique(axis=1)
print(f"rows where all 5 example_idx are distinct: {(n_distinct == 5).sum()}/{len(df_selected)}")

print("\ndistractor reuse check per distractor_bin (slots filled vs. distinct docs used):")
for (d_low, d_high), group in df_selected.groupby(["distractor_bin_low", "distractor_bin_high"]):
    used = []
    for _, row in group.iterrows():
        for label in LABELS:
            if label != row["correct_label"]:
                used.append(row[f"example_idx_{label}"])
    flag = "" if len(used) == len(set(used)) else "  <-- REUSE DETECTED"
    print(f"  [{d_low},{d_high}]: {len(used)} slots, {len(set(used))} distinct docs{flag}")

rows where all 5 example_idx are distinct: 3200/3200

distractor reuse check per distractor_bin (slots filled vs. distinct docs used):
  [0,3]: 1600 slots, 1600 distinct docs
  [3,7]: 1600 slots, 1600 distinct docs
  [7,19]: 1600 slots, 1600 distinct docs
  [19,50]: 1600 slots, 1600 distinct docs
  [50,81]: 1600 slots, 1600 distinct docs
  [81,93]: 1600 slots, 1600 distinct docs
  [93,97]: 1600 slots, 1600 distinct docs
  [97,100]: 1600 slots, 1600 distinct docs


In [6]:
# Sanity check 3 (the substantive one): does each candidate's ACTUAL score
# percentile -- looked up from df by (example_idx, position, token_id), the same
# ground truth the selection script itself computed from -- actually fall within
# the bin it was supposedly selected for? This directly verifies the selection
# script put the right tokens in the right bins, rather than just trusting it.
# Same boundary convention as the selection script's _in_bin helper: the very
# first bin (low == bins_edges[0]) is closed on both ends, every other bin is
# left-open/right-closed.
lookup = df.set_index(["example_idx", "position", "token_id"])["pctile"]

mismatches = []
for _, row in df_selected.iterrows():
    for label in LABELS:
        key = (row[f"example_idx_{label}"], row[f"position_{label}"], row[f"token_id_{label}"])
        actual_pctile = lookup.loc[key]
        is_intruder = label == row["correct_label"]
        lo, hi = (row["intruder_bin_low"], row["intruder_bin_high"]) if is_intruder \
            else (row["distractor_bin_low"], row["distractor_bin_high"])
        in_range = (actual_pctile >= lo if lo == bins_edges[0] else actual_pctile > lo) and actual_pctile <= hi
        if not in_range:
            mismatches.append((label, actual_pctile, lo, hi, "intruder" if is_intruder else "distractor"))

print(f"{len(mismatches)} bin mismatches out of {len(df_selected) * 5} candidate slots checked")
for label, actual_pctile, lo, hi, role in mismatches[:10]:
    print(f"  MISMATCH: label={label} ({role})  actual_pctile={actual_pctile:.2f}  expected [{lo},{hi}]")

0 bin mismatches out of 16000 candidate slots checked


In [7]:
# Cross-doc judge smoke test results (judge_cross_doc_intruder.py), pulled from the cluster.
df_results = pd.read_csv("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/results_ib_0_1.csv")
print(df_results.shape)
print(df_results[["side", "distractor_bin_low", "distractor_bin_high", "correct_label",
                   "predicted_label", "is_correct", "truncated"]])
print(f"\naccuracy: {df_results['is_correct'].mean():.2f}")
print(f"truncated: {df_results['truncated'].sum()}/{len(df_results)}")

(50, 10)
   side  distractor_bin_low  distractor_bin_high correct_label  \
0   top                   0                    1             D   
1   top                   0                    1             C   
2   top                   0                    1             B   
3   top                   0                    1             A   
4   top                   0                    1             B   
5   top                   1                    2             D   
6   top                   1                    2             B   
7   top                   1                    2             D   
8   top                   1                    2             C   
9   top                   1                    2             D   
10  top                   2                    7             D   
11  top                   2                    7             C   
12  top                   2                    7             C   
13  top                   2                    7             E   
1

In [8]:
print(pd.crosstab(df_results["truncated"], df_results["predicted_label"].isna()))
df_results

predicted_label  False  True 
truncated                    
False               49      0
True                 0      1


,side,intruder_bin_low,intruder_bin_high,distractor_bin_low,distractor_bin_high,correct_label,predicted_label,truncated,completion_tokens,is_correct
0,top,0.0,1.0,0,1,D,E,False,827,False
1,top,0.0,1.0,0,1,C,D,False,561,False
2,top,0.0,1.0,0,1,B,B,False,559,True
3,top,0.0,1.0,0,1,A,B,False,723,False
4,top,0.0,1.0,0,1,B,B,False,671,True
5,top,0.0,1.0,1,2,D,C,False,584,False
6,top,0.0,1.0,1,2,B,E,False,519,False
7,top,0.0,1.0,1,2,D,E,False,674,False
8,top,0.0,1.0,1,2,C,A,False,3769,False
9,top,0.0,1.0,1,2,D,D,False,495,True


In [ ]:
# Same checks as above (shape, accuracy, truncated/unanswered crosstab), looped
# over every v2 smoke test result file (one per intruder bin) pulled into smoketest_v2/.
smoketest_v2_dir = Path("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/smoketest_v2")
result_files = sorted(smoketest_v2_dir.glob("*.csv"))

for f in result_files:
    df_r = pd.read_csv(f)
    print("=" * 80)
    print(f"{f.name}: {df_r.shape}")
    print(f"accuracy: {df_r['is_correct'].mean():.2f}")
    print(f"truncated: {df_r['truncated'].sum()}/{len(df_r)}")
    print(pd.crosstab(df_r["truncated"], df_r["predicted_label"].isna()))

In [ ]:
# 1D slice per file: each file is one fixed intruder bin, so within it accuracy can
# only vary over distractor_bin -- this is exactly one row of the eventual 2D
# (intruder_bin x distractor_bin) grid. One plot per file; combine into the 2D
# heatmap once there's more than a smoke-test's worth of rows per bin to trust.
# 95% CI per bin via binomtest, same technique as analyse_intruder_experiment.ipynb's
# per-bin accuracy plot.
for f in result_files:
    df_r = pd.read_csv(f)
    side = df_r["side"].iloc[0]
    ib_low, ib_high = df_r["intruder_bin_low"].iloc[0], df_r["intruder_bin_high"].iloc[0]

    labels, accuracies, err_low, err_high, ns = [], [], [], [], []
    for (d_lo, d_hi), group in df_r.groupby(["distractor_bin_low", "distractor_bin_high"]):
        n = len(group)
        n_correct = int(group["is_correct"].sum())
        acc = n_correct / n
        ci = binomtest(n_correct, n).proportion_ci(confidence_level=0.95)
        labels.append(f"{d_lo}-{d_hi}")
        accuracies.append(acc)
        ns.append(n)
        err_low.append(acc - ci.low)
        err_high.append(ci.high - acc)

    plt.figure(figsize=(6, 4))
    plt.errorbar(labels, accuracies, yerr=[err_low, err_high], fmt="o-", capsize=3)
    plt.axhline(0.2, color="gray", linestyle="--", label="chance (0.2)")
    plt.ylim(0, 1)
    plt.xlabel("distractor bin (percentile range)")
    plt.ylabel("accuracy")
    plt.title(f"{f.name}\nside={side}, intruder bin [{ib_low}, {ib_high}] "
              f"(n={sum(ns)} total, {min(ns)}-{max(ns)} per bin)")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 2D grid: intruder_bin (rows) x distractor_bin (cols) accuracy, combining across
# files. Only files sharing the SAME --bin-edges grid can be combined -- the very
# first smoke test (results.csv, intruder bin [0,3]) used the older/coarser edges
# [0,3,7,19,50,81,93,97,100], not this grid's [0,1,2,7,19,50,81,93,98,99,100], so
# it's skipped rather than silently misplaced into a cell it doesn't match.
bin_edges_v2 = [0, 1, 2, 7, 19, 50, 81, 93, 98, 99, 100]
bin_pairs = list(zip(bin_edges_v2[:-1], bin_edges_v2[1:]))

frames = []
for f in result_files:
    df_r = pd.read_csv(f)
    ib_low, ib_high = df_r["intruder_bin_low"].iloc[0], df_r["intruder_bin_high"].iloc[0]
    if (ib_low, ib_high) not in bin_pairs:
        print(f"Skipping {f.name}: intruder bin [{ib_low},{ib_high}] doesn't match this grid's bin edges {bin_edges_v2}")
        continue
    frames.append(df_r)
df_all = pd.concat(frames, ignore_index=True)

acc = df_all.groupby(["intruder_bin_low", "intruder_bin_high",
                       "distractor_bin_low", "distractor_bin_high"])["is_correct"].mean()

n_bins = len(bin_pairs)
grid = np.full((n_bins, n_bins), np.nan)
for i, (i_lo, i_hi) in enumerate(bin_pairs):
    for j, (d_lo, d_hi) in enumerate(bin_pairs):
        key = (i_lo, i_hi, d_lo, d_hi)
        if key in acc.index:
            grid[i, j] = acc.loc[key]

labels = [f"{lo}-{hi}" for lo, hi in bin_pairs]
masked = np.ma.masked_invalid(grid)
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("lightgray")

plt.figure(figsize=(8, 7))
im = plt.imshow(masked, cmap=cmap, vmin=0, vmax=1)
plt.colorbar(im, label="accuracy")
plt.xticks(range(n_bins), labels, rotation=45)
plt.yticks(range(n_bins), labels)
plt.xlabel("distractor bin (percentile range)")
plt.ylabel("intruder bin (percentile range)")
plt.title("2D accuracy grid (side=top) -- smoke test, small n per cell\n"
          "gray = no data (either below the diagonal, or that intruder bin not yet run)")
for i in range(n_bins):
    for j in range(n_bins):
        if not np.isnan(grid[i, j]):
            plt.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center",
                      color="white" if grid[i, j] < 0.5 else "black", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Row-normalized confusion matrix, pooled across ALL smoke test files (unlike the
# 2D accuracy grid above, this doesn't need matching bin edges -- it's checking for
# positional bias in the judge/prompt itself, which shouldn't depend on bin
# difficulty. Per-file it'd only be ~10 calls per label, too thin to read; pooling
# across all 6 files (~280 calls) still isn't much, but it's the best this smoke
# test can offer.
df_combined = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)
CM_LABELS = list("ABCDE")

confusion = pd.crosstab(df_combined["correct_label"], df_combined["predicted_label"])
confusion = confusion.reindex(index=CM_LABELS, columns=CM_LABELS, fill_value=0)
confusion_norm = confusion.div(confusion.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(confusion_norm.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(confusion_norm.columns)))
ax.set_xticklabels(confusion_norm.columns)
ax.set_yticks(range(len(confusion_norm.index)))
ax.set_yticklabels(confusion_norm.index)
ax.set_xlabel("predicted_label")
ax.set_ylabel("correct_label")
ax.set_title(f"Confusion matrix (row-normalized): P(predicted | correct)\n"
             f"pooled across all {len(result_files)} smoke test files, n={len(df_combined)}")

for i in range(confusion_norm.shape[0]):
    for j in range(confusion_norm.shape[1]):
        value = confusion_norm.values[i, j]
        color = "white" if value > 0.5 else "black"
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", color=color)

fig.colorbar(im, ax=ax, label="fraction of row")
plt.tight_layout()
plt.show()